<a href="https://colab.research.google.com/github/brendonhuynhbp-hub/gt-markets/blob/main/notebooks/50_build_app_artefacts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# Build artefacts from extracted runs (keywords models) — clean, aligned
# - Reads prediction CSVs from app-demo/extracted/{daily,weekly}
# - Derives signals/positions and equity curves
# - Writes metrics_keywords_{D|W}.csv, signals_*.csv, figs/*.png
# - Index alignment fixed to avoid pandas warnings
# ============================================================

# Mount Drive (idempotent in Colab)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import math, re

# --- Paths ---
PROJECT_DIR   = Path("/content/drive/MyDrive/gt-markets")
SRC_EXTRACT   = PROJECT_DIR / "app-demo" / "extracted"   # daily/weekly folders
DST_ARTE      = PROJECT_DIR / "AppDemo"  / "artefacts"   # merge with baseline outputs
DATA_PROCESSED= PROJECT_DIR / "data"     / "processed"

ASSETS   = ["GOLD", "BTC", "OIL", "USDCNY"]
FREQ_DIR = {"daily": "D", "weekly": "W"}
ANNUAL   = {"D": 252, "W": 52}
COST_BPS = 5  # one-way bps per position change

# Map to processed prices for Close fallback
PRICE_COL = {
    "GOLD":   "GC=F Close",
    "BTC":    "BTC-USD Close",
    "OIL":    "CL=F Close",
    "USDCNY": "USDCNY=X Close",
}

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def to_equity(close: pd.Series, pos: pd.Series, freq: str):
    """
    Equity curve + metrics using aligned DateTimeIndex. Assumes:
      - close: Series of prices indexed by Date
      - pos:   Series of {0,1} exposure indexed by same Date
    Index alignment is enforced here to avoid pandas reindex warnings.
    """
    # Normalize dtype and index
    close = pd.Series(pd.to_numeric(close, errors="coerce"))
    close.index = pd.to_datetime(close.index)

    pos = pd.Series(pd.to_numeric(pos, errors="coerce"))
    pos.index = close.index                   # align to price timeline
    pos = pos.ffill().fillna(0.0).clip(0, 1)  # allow occasional gaps

    # Returns with explicit fill_method=None (no implicit padding)
    ret = close.pct_change(fill_method=None).fillna(0.0)

    trades = pos.diff().abs().fillna(0.0)
    cost   = trades * (COST_BPS / 1e4)

    strat = (pos.shift(1).fillna(0.0) * ret) - cost
    eq    = (1.0 + strat).cumprod()

    ann = ANNUAL[freq]
    mu  = strat.mean() * ann
    sd  = strat.std()  * math.sqrt(ann)
    sr  = mu / sd if sd > 0 else np.nan
    mdd = (eq / eq.cummax() - 1.0).min()
    return eq, {"Return_Ann": mu, "Vol_Ann": sd, "Sharpe": sr, "MaxDD": mdd}

def infer_asset(s: str):
    s = s.lower()
    if "gold" in s or "gc=f" in s:   return "GOLD"
    if "btc"  in s or "bitcoin" in s:return "BTC"
    if "oil"  in s or "cl=f" in s:   return "OIL"
    if "usdcny" in s:                return "USDCNY"
    return None

def is_prediction_like(p: Path, df: pd.DataFrame) -> bool:
    """Heuristics to only process model outputs, not leaderboards/metrics."""
    name = p.name.lower()
    if any(tok in name for tok in ["pred", "prob", "proba", "inference", "signals"]):
        return True
    cols = {c.lower() for c in df.columns}
    return ("prob_up" in cols) or ("p_up" in cols) or ("position" in cols)

def clean_strategy_name(p: Path) -> str:
    """
    Build a readable strategy id from parent/stem.
    Removes window tags (w30), generic tokens, keeps model token if present.
    """
    toks = re.split(r"[_\-\.\s]+", (p.parent.name + "_" + p.stem).lower())
    drop = {"daily","weekly","test","pred","prediction","proba","prob","signals","val","fold"}
    toks = [t for t in toks if not re.fullmatch(r"w\d+", t) and t not in drop and t]

    # prefer known model identifiers
    preferred = ["xgb","xgboost","rf","randforest","randomforest","lr","logreg","mlp","lstm","gru","svm"]
    for k in preferred:
        if k in toks:
            return "KW_" + (k.upper() if len(k) <= 4 else k)

    # fallback to first meaningful token
    return "KW_" + (toks[0].upper() if toks else "MODEL")

def load_price_fallback() -> pd.DataFrame | None:
    """Load the latest merged processed price file for Close fallback."""
    files = sorted(DATA_PROCESSED.glob("merged_financial_trends_data_*.csv"))
    if not files:
        return None
    try:
        df = pd.read_csv(files[-1], parse_dates=["Date"]).set_index("Date").sort_index()
        return df
    except Exception:
        return None

price_df = load_price_fallback()

# ------------------------------------------------------------
# Main loop
# ------------------------------------------------------------

summary = []  # (sub, file, asset, status, note/strategy)

for sub, F in FREQ_DIR.items():
    src = SRC_EXTRACT / sub
    if not src.exists():
        summary.append((sub, "-", "-", "SKIP", "folder missing"))
        continue

    for csv in src.rglob("*.csv"):
        # Skip obvious non-prediction tables
        lname = csv.name.lower()
        if any(bad in lname for bad in ["metrics", "leaderboard", "summary", "baseline", "keywords"]):
            continue

        asset = infer_asset(csv.name) or infer_asset(str(csv))
        if asset is None:
            summary.append((sub, csv.name, "-", "SKIP", "unknown asset"))
            continue

        try:
            df = pd.read_csv(csv)
        except Exception as e:
            summary.append((sub, csv.name, asset, "FAIL", f"read_error: {e}"))
            continue

        # Normalize datetime column
        if "Date" not in df.columns and "date" in df.columns:
            df = df.rename(columns={"date": "Date"})
        if "Date" not in df.columns:
            summary.append((sub, csv.name, asset, "SKIP", "no Date col"))
            continue

        if not is_prediction_like(csv, df):
            summary.append((sub, csv.name, asset, "SKIP", "not prediction-like"))
            continue

        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        df = df.dropna(subset=["Date"]).sort_values("Date")

        # Position: prefer provided "position", else threshold on prob_up-like
        pos = None
        for c in ["position", "Position"]:
            if c in df.columns:
                pos = pd.to_numeric(df[c], errors="coerce")
                break
        if pos is None:
            pcol = next((c for c in ["prob_up","p_up","proba","prob","prob1","p1"] if c in df.columns), None)
            if pcol is None:
                summary.append((sub, csv.name, asset, "SKIP", "no position/prob"))
                continue
            pos = (pd.to_numeric(df[pcol], errors="coerce") > 0.5).astype(float)

        # Close: use in-file price or fallback to processed prices
        close = None
        ccol  = next((c for c in ["Close","close","price","close_adj","adj_close"] if c in df.columns), None)
        if ccol is not None:
            close = pd.to_numeric(df[ccol], errors="coerce")
        elif price_df is not None:
            psrc = PRICE_COL.get(asset)
            if psrc and psrc in price_df.columns:
                close = price_df[psrc].reindex(df["Date"]).ffill()

        if close is None:
            summary.append((sub, csv.name, asset, "SKIP", "no Close + no fallback"))
            continue

        # Align indices explicitly to silence pandas warnings
        close.index = df["Date"]
        pos.index   = df["Date"]

        strat = clean_strategy_name(csv)

        # --- write signals file ---
        out_dir = DST_ARTE / asset
        (out_dir / "figs").mkdir(parents=True, exist_ok=True)

        # Signal = 1 on buy, 0 on sell (from position diff)
        sig = pos.diff().fillna(0.0).replace({1.0: 1, -1.0: 0}).astype(int)

        pd.DataFrame({
            "Date":     df["Date"].values,
            "signal":   sig.values,
            "position": pos.astype(int).values,
            "Close":    pd.to_numeric(close, errors="coerce").values,
            "strategy": strat
        }).to_csv(out_dir / f"signals_{strat}_{F}.csv", index=False)

        # --- metrics + equity fig ---
        eq, mets = to_equity(close, pos, F)
        ax = eq.plot(figsize=(6, 3), title=f"{asset} — {strat} — {F}")
        ax.grid(True, alpha=0.3)
        ax.get_figure().savefig(out_dir / "figs" / f"{strat}_{F}.png", dpi=150, bbox_inches="tight")
        ax.get_figure().clf()
        plt.close(ax.get_figure())

        row = {"asset": asset, "freq": F, "strategy": strat, **mets}
        out_csv = out_dir / f"metrics_keywords_{F}.csv"
        if out_csv.exists():
            old = pd.read_csv(out_csv)
            pd.concat([old, pd.DataFrame([row])], ignore_index=True).to_csv(out_csv, index=False)
        else:
            pd.DataFrame([row]).to_csv(out_csv, index=False)

        summary.append((sub, csv.name, asset, "OK", strat))

# ------------------------------------------------------------
# Log summary (first 200 lines to keep console readable)
# ------------------------------------------------------------
print("Summary (folder, file, asset, status, note/strategy):")
for s in summary[:200]:
    print("  -", s)

print("\n✅ Keyword artefacts merged into:", DST_ARTE)


Mounted at /content/drive
Summary (folder, file, asset, status, note/strategy):
  - ('daily', 'dataset_snapshot_merged_financial_trends_data_2025-09-07.csv.head1.csv', '-', 'SKIP', 'unknown asset')
  - ('daily', 'oil_eng_ext_lstm_w30.val_f0.csv', 'OIL', 'OK', 'KW_LSTM')
  - ('daily', 'usdcny_eng_ext_gru_w30.val_f0.csv', 'USDCNY', 'OK', 'KW_GRU')
  - ('daily', 'usdcny_eng_ext_rf.val_f0.csv', 'USDCNY', 'OK', 'KW_RF')
  - ('daily', 'oil_eng_ext_gru_w30.val_f0.csv', 'OIL', 'OK', 'KW_GRU')
  - ('daily', 'usdcny_eng_ext_lr.val_f0.csv', 'USDCNY', 'OK', 'KW_LR')
  - ('daily', 'usdcny_eng_ext_xgb.val_f0.csv', 'USDCNY', 'OK', 'KW_XGB')
  - ('daily', 'oil_eng_ext_lr.val_f0.csv', 'OIL', 'OK', 'KW_LR')
  - ('daily', 'btc_eng_ext_lstm_w30.val_f0.csv', 'BTC', 'OK', 'KW_LSTM')
  - ('daily', 'btc_eng_ext_gru_w30.val_f0.csv', 'BTC', 'OK', 'KW_GRU')
  - ('daily', 'oil_eng_ext_xgb.val_f0.csv', 'OIL', 'OK', 'KW_XGB')
  - ('daily', 'btc_eng_ext_xgb.val_f0.csv', 'BTC', 'OK', 'KW_XGB')
  - ('daily', 'oil_eng_

In [3]:
# --- Normalise artefact folders to uppercase ---
from pathlib import Path
import shutil, os

ARTE = Path("/content/drive/MyDrive/gt-markets/AppDemo/artefacts")

pairs = [("Gold","GOLD"), ("Oil","OIL")]
for src_name, dst_name in pairs:
    src, dst = ARTE/src_name, ARTE/dst_name
    if src.exists():
        dst.mkdir(parents=True, exist_ok=True)
        for p in src.iterdir():
            target = dst / p.name
            if target.exists():
                backup = dst / (p.stem + ".old" + p.suffix)
                print(f"⚠️ {target.name} already in {dst_name}, backing up {p.name} as {backup.name}")
                shutil.move(str(p), str(backup))
            else:
                shutil.move(str(p), str(dst))
        os.rmdir(src)
        print(f"✅ Merged {src_name} → {dst_name}")

# Quick validation
good = {"BTC","GOLD","OIL","USDCNY"}
bad  = [d.name for d in ARTE.iterdir() if d.is_dir() and d.name not in good]
if bad:
    print("❌ Found non-canonical folders:", bad)
else:
    print("✅ Artefact folders look clean:", list(good))


⚠️ metrics_baseline_W.csv already in GOLD, backing up metrics_baseline_W.csv as metrics_baseline_W.old.csv
⚠️ figs already in GOLD, backing up figs as figs.old
⚠️ metrics_keywords_W.csv already in GOLD, backing up metrics_keywords_W.csv as metrics_keywords_W.old.csv
⚠️ metrics_baseline_D.csv already in GOLD, backing up metrics_baseline_D.csv as metrics_baseline_D.old.csv
⚠️ metrics_keywords_D.csv already in GOLD, backing up metrics_keywords_D.csv as metrics_keywords_D.old.csv
✅ Merged Gold → GOLD
⚠️ figs already in OIL, backing up figs as figs.old
⚠️ metrics_keywords_W.csv already in OIL, backing up metrics_keywords_W.csv as metrics_keywords_W.old.csv
⚠️ metrics_baseline_W.csv already in OIL, backing up metrics_baseline_W.csv as metrics_baseline_W.old.csv
⚠️ metrics_baseline_D.csv already in OIL, backing up metrics_baseline_D.csv as metrics_baseline_D.old.csv
⚠️ metrics_keywords_D.csv already in OIL, backing up metrics_keywords_D.csv as metrics_keywords_D.old.csv
✅ Merged Oil → OIL
✅ A